In [2]:
import sys
sys.path.append("..")
from src.genai.llm_client import ask_llm

print(ask_llm("You are a helpful assistant.", "Say hello world in one short sentence."))

Hello, world!


In [3]:
import sys
sys.path.append("..")
from src.database.db_connection import load_config
print(load_config()["prompts"])

{'active_delay_briefing_version': 'v1', 'active_delay_whatif_version': 'v1', 'active_shortage_briefing_version': 'v1', 'active_shortage_whatif_version': 'v1'}


In [4]:
import sys
sys.path.append("..")
from src.database import queries
from src.genai.prompt_templates import build_prompt
from src.genai.llm_client import ask_llm

# Delay briefing
facts = queries.get_delay_facts(2)
system, user = build_prompt("delay_briefing", facts)
print(user)
print("--- LLM ANSWER ---")
print(ask_llm(system, user))

# Shortage briefing
facts = queries.get_shortage_facts(176147)
system, user = build_prompt("shortage_briefing", facts)
print(user)
print("--- LLM ANSWER ---")
print(ask_llm(system, user))

# What-if examples
facts = queries.get_delay_facts(2)
system, user = build_prompt("delay_whatif", facts, question="What if we ship this order with First Class instead?")
print(ask_llm(system, user))

facts = queries.get_shortage_facts(176147)
system, user = build_prompt("shortage_whatif", facts, question="What if demand for this product rises by 20% next week?")
print(ask_llm(system, user))

Order details:
- Shipping mode: Standard Class
- Region: South America
- Category: Water Sports
- Predicted delay probability: 41%
- Risk level: Low

Historical numbers (from orders before this one):
- Shipping mode average delay: -0.01 days (std 1.42), reliability score -2.00
- Region average delay: 0.56 days (std 1.51), reliability score 0.57

Write a briefing in exactly 3 short sentences:
1. State the risk level and the delay probability.
2. Explain the main reason, using the numbers above.
3. Suggest one practical action.
--- LLM ANSWER ---
Risk level is Low with a 41% delay probability. The South America region shows a positive average delay of 0.56 days, while the Standard Class mode is near zero (-0.01 days) but has high variability (std 1.42), making the region the dominant factor. Monitor the shipment closely and be ready to expedite if delays arise.
Product details:
- Product: Adult dog supplies (category: Pet Supplies)
- Shortage risk score: 0.72 (higher = riskier). This is 

In [5]:
import sys
sys.path.append("..")
from src.database import queries
from src.genai.prompt_templates import build_prompt
from src.genai.llm_client import ask_llm

delay_facts = queries.get_delay_facts(2)
delay_facts.update(queries.get_delay_context())
shortage_facts = queries.get_shortage_facts(176147)

tests = [
    ("delay_briefing",    delay_facts,    None),
    ("delay_whatif",      delay_facts,    "What if we ship this order with First Class instead?"),
    ("shortage_briefing", shortage_facts, None),
    ("shortage_whatif",   shortage_facts, "What if demand for this product rises by 20% next week?"),
]

for kind, facts, question in tests:
    for version in ["v1", "v2"]:
        system, user = build_prompt(kind, facts, question, version=version)
        print(f"=== {kind} | {version} ===")
        print(ask_llm(system, user))
        print()

=== delay_briefing | v1 ===
Risk level is Low with a 41% probability of delay. The region’s average delay of 0.56 days (reliability 0.57) outweighs the near‑zero shipping‑mode delay of –0.01 days (reliability –2.00), driving the higher delay chance. Monitor and adjust the South America routing schedule to add a buffer of one day.

=== delay_briefing | v2 ===
The order is Low risk with a 41% delay probability, which is below the overall late rate of 57.3%. Because Standard Class has a near‑zero average delay (‑0.01 days) and South America shows only a modest 0.56‑day delay, the risk remains low. Monitor the shipment daily and confirm carrier pickup times.

=== delay_whatif | v1 ===
Switching to First Class would change the delay profile, because the current average delay of ‑0.01 days applies only to Standard Class. Without a First‑Class average delay value, we can’t tell whether the risk will rise or fall. The model would need to be re‑run with First‑Class data to get a new probability

In [1]:
import sys
sys.path.append("..")
from src.database import queries
from src.genai.prompt_templates import build_prompt, format_lines
from src.genai.llm_client import ask_llm

delay_facts = queries.get_delay_facts(2)
delay_facts.update(queries.get_delay_context())
shortage_facts = queries.get_shortage_facts(176147)

tests = [
    ("delay_briefing",    delay_facts,    None),
    ("delay_whatif",      delay_facts,    "What if we ship this order with First Class instead?"),
    ("shortage_briefing", shortage_facts, None),
    ("shortage_whatif",   shortage_facts, "What if demand for this product rises by 20% next week?"),
]

for kind, facts, question in tests:
    system, user = build_prompt(kind, facts, question, version="v3")
    answer = format_lines(ask_llm(system, user))
    print("=" * 60)
    print(kind.upper().replace("_", " "))
    print("=" * 60)
    print(answer)
    print()

DELAY BRIEFING
Risk level is Low with a 41% delay probability, which is below the overall late rate.
The Standard Class mode (average delay -0.01 days) and South America region (average delay 0.56 days) both show minimal delays, supporting the Low risk.
Monitor shipment progress daily and inform the customer of any updates.

DELAY WHATIF
Standard Class currently has a late rate of 39.8%.
First Class would have a late rate of 100.0%, which is much higher.
The overall late rate is 57.3%, so Standard is below and First is above it.
The model must be re‑run for an exact value.

SHORTAGE BRIEFING
The risk level is High.
The limited history of only one past order makes the high score less reliable, with average demand of 237.9 units and a standard deviation of 30.2.
Consider keeping safety stock equal to one standard deviation (30.2 units) above expected demand.

SHORTAGE WHATIF
The average weekly demand would rise to about 285.5 units.
This is an increase of roughly 47.6 units over the curr

In [1]:
import sys, time
sys.path.append("..")
from src.genai.briefing import get_explanation

for i in range(2):
    t = time.time()
    r = get_explanation("delay_briefing", 2)
    print(f"Call {i+1}: cached={r['cached']}, time={time.time()-t:.2f}s")
    print(r["text"])
    print()

r = get_explanation("shortage_whatif", 176147, question="What if demand rises by 20% next week?")
print(r["cached"])
print(r["text"])

Call 1: cached=False, time=3.14s
Risk level is Low with a 41% delay probability, which is below the overall late rate.
The Standard Class mode has an average delay of -0.01 days and the South America region averages 0.56 days, keeping risk low.
Track the shipment daily and arrange expedited shipping if a delay emerges.

Call 2: cached=True, time=0.01s
Risk level is Low with a 41% delay probability, which is below the overall late rate.
The Standard Class mode has an average delay of -0.01 days and the South America region averages 0.56 days, keeping risk low.
Track the shipment daily and arrange expedited shipping if a delay emerges.

False
The average weekly demand would rise to about 285.5 units.
This is an increase of roughly 47.6 units over the current average.
The increase exceeds the demand variability, being about 1.6 times the standard deviation of 30.2.
The model must be re-run for an exact value.


In [2]:
import pandas as pd
from src.database.db_connection import get_connection
conn = get_connection()
print(pd.read_sql("SELECT id, kind, entity_key, prompt_version, question, created_at FROM llm_briefings", conn))
conn.close()

   id             kind entity_key prompt_version  \
0   1   delay_briefing          2             v3   
1   2  shortage_whatif     176147             v3   

                                 question           created_at  
0                                     NaN  2026-10-26 19:14:02  
1  What if demand rises by 20% next week?  2026-10-26 19:14:04  


In [2]:
import sys
sys.path.append("..")
from src.database.db_connection import load_config
from src.genai.prompt_templates import PROMPTS

print(load_config()["prompts"])
print("delay_whatif versions:   ", list(PROMPTS["delay_whatif"].keys()))
print("shortage_whatif versions:", list(PROMPTS["shortage_whatif"].keys()))

{'active_delay_briefing_version': 'v3', 'active_delay_whatif_version': 'v4', 'active_shortage_briefing_version': 'v3', 'active_shortage_whatif_version': 'v4'}
delay_whatif versions:    ['v1', 'v2', 'v3', 'v4']
shortage_whatif versions: ['v1', 'v2', 'v3', 'v4']


In [3]:
import sys
sys.path.append("..")
from src.database import queries
from src.genai.prompt_templates import build_prompt, format_lines
from src.genai.llm_client import ask_llm

delay_facts = queries.get_delay_facts(2)
delay_facts.update(queries.get_delay_context())
shortage_facts = queries.get_shortage_facts(176147)

tests = [
    ("delay_whatif",    delay_facts,    "What if this order is sent to Western Europe instead?"),
    ("delay_whatif",    delay_facts,    "What if the delivery takes 2 days longer than scheduled?"),
    ("shortage_whatif", shortage_facts, "What if demand for this product rises by 20% next week?"),
    ("shortage_whatif", shortage_facts, "What if demand drops by 30%?"),
]

for kind, facts, question in tests:
    print("=" * 70)
    print("Q:", question)
    for version in ["v3", "v4"]:
        system, user = build_prompt(kind, facts, question, version=version)
        answer = format_lines(ask_llm(system, user))
        print(f"--- {version} ---")
        print(answer)
    print()

Q: What if this order is sent to Western Europe instead?
--- v3 ---
The Standard Class late rate is 39.8%, well below the overall 57.3% late rate.
If the order goes to Western Europe, the shipping mode stays Standard, so the same 39.8% rate applies.
This rate is lower than First Class (100%), Second Class (79.8%) and Same Day (47.9%).
The model must be re‑run for an exact value.
--- v4 ---
The order’s current region South America has a 57.2% late rate.
Western Europe has a 58.5% late rate, which is higher than South America and the overall 57.3% rate.
The model must be re‑run for an exact value.

Q: What if the delivery takes 2 days longer than scheduled?
--- v3 ---
The Standard Class mode currently has a 39.8% late rate.
The overall late rate for all orders is 57.3%.
Adding a 2‑day delay would raise the risk above the Standard Class rate and likely exceed the overall rate.
The model must be re‑run for an exact value.
--- v4 ---
The extra 2 days exceeds the standard deviation of 1.42 d